# From gates to circuits

The Bell pair in `intro.py` was my starting point. I like to pause just before its measurement and account for every branch.

In [1]:
from math import pi

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

## Bell pair

After H on q0 I have 00 and 01 with equal amplitude. CX leaves 00 alone and sends 01 to 11. Predict which two slots in the probability array will be empty.

In [2]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
print(bell.draw(output="text"))
print("Bell probabilities:", Statevector(bell).probabilities().round(3))

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
Bell probabilities: [0.5 0.  0.  0.5]


Matching bits are only part of the story. A classical mixture of 00 and 11 could also match every time. To see the phase coherence in the Bell pair, undo the preparation. Both branches should recombine into 00. A classical mixture would still give two possible outcomes after undoing the gates.

In [3]:
decoded = bell.compose(bell.inverse())
print("After undoing:", Statevector(decoded).probabilities().round(3))

After undoing: [1. 0. 0. 0.]


## Three friends sharing a bit

One more CX extends the same idea to a GHZ state. Each wire looks random on its own. Once I read the first bit in this basis, I can predict the other two.

In [4]:
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
state = Statevector(ghz)
state.seed(19)
print(ghz.draw(output="text"))
print("GHZ counts:", dict(state.sample_counts(100)))

     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘
GHZ counts: {np.str_('000'): np.int64(45), np.str_('111'): np.int64(55)}


## Phase kickback

Here I prepare the target in |−⟩. X sends |−⟩ to −|−⟩, so the branch where the control is 1 picks up a minus sign. That turns the control from |+⟩ into |−⟩. Guess what the final H will reveal.

In [5]:
kick = QuantumCircuit(2)
kick.h(0)                 # control in |+>
kick.x(1)
kick.h(1)                 # target in |->
kick.cx(0, 1)
kick.h(0)                 # read the control in the +/- basis
print("Control probabilities:", Statevector(kick).probabilities([0]).round(3))

Control probabilities: [0. 1.]


Try removing the X that prepares the target. Now the target is |+⟩, which X leaves unchanged. The final control should read 0. This trick of writing information into a phase turns up again in search algorithms.